In [ ]:
import numpy as np
from PMM.PMMInverse import PMMI
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 1000
import os

In [ ]:
a = 0.0215
res = 30
nx = 19
ny = 24
dpml = 2
b_o = 0.0075/a
b_i = 0.0065/a
entrance = 0.04/a
output = os.getcwd()+'/../outputs'
fname = '6by6HexWvg_ez_w025_wpmax047_gam0GHz_res50_coldstart'
run_no = ['_r0','_r1']

In [ ]:
## Set up domain geometry #####################################################
PPC = PMMI(a, res, nx, ny, dpml) #Initialize PMMI object
PPC.Add_INFOMW_Horn(np.array([6.5, 12]), np.array([1,0]), 6.5, pol ='TM')
PPC.Add_INFOMW_Horn(5*np.array([0.5, 3**0.5/2])+np.array([10.5+np.sqrt(3)/2,12]), np.array([-0.5,-3**0.5/2]), 8, pol ='TM')
PPC.Add_INFOMW_Horn(5*np.array([0.5, -3**0.5/2])+np.array([10.5+np.sqrt(3)/2,12]), np.array([-0.5,3**0.5/2]), 8, pol ='TM')
PPC.Design_Region((6.5,6.5), (11.5*3**0.5/2, 11)) #Specify Region where elements are being optimized

uniform = True
PPC.Rod_Array_Hexagon_train(np.array([10.5+np.sqrt(3)/2,12]), 6, b_i, 1,\
                          a_basis = np.array([[0,1],[np.sqrt(3)/2,1./2]]),\
                          bulbs = True, r_bulb = (b_i, b_o), eps_bulb = 3.8,
                          uniform = uniform) #Rod ppc array


In [ ]:
## Set up Sources and Sim #####################################################
w = PPC.gamma(6.001e9) #Source frequency
wpmax = PPC.gamma(10e9)
gamma = 0#PPC.gamma(1e9)

ew = 0.048/a/2-0.004/a
hd = 0.089/a
x = np.array([1,0])
y = np.array([0,1])
horn_dir_1 = np.array([0.5,3**0.5/2])
horn_dir_2 = np.array([0.5,-3**0.5/2])
open_dir_1 = np.array([3**0.5/2,-0.5])
open_dir_2 = np.array([3**0.5/2,0.5])
cen = np.array([10.5+np.sqrt(3)/2,12])

PPC.Add_Source(np.array([6.5-hd,12-ew]), np.array([6.5-hd,12+ew]), w, 'src', 'ez')
PPC.Add_Probe((5+hd)*horn_dir_1 + ew*open_dir_1 + cen,\
              (5+hd)*horn_dir_1 - ew*open_dir_1 + cen, w, 'prb1', 'ez')
PPC.Add_Probe((5+hd)*horn_dir_2 + ew*open_dir_2 + cen,\
              (5+hd)*horn_dir_2 - ew*open_dir_2 + cen, w, 'prb2', 'ez')
PPC.Add_Probe(-5.6*(3**0.5/2)*horn_dir_1 + 2.75*open_dir_1 + cen,\
              -5.6*(3**0.5/2)*horn_dir_1 - 2.75*open_dir_1 + cen, w, 'loss_ul', 'ez')
PPC.Add_Probe(-5.6*(3**0.5/2)*horn_dir_2 + 2.75*open_dir_2 + cen,\
              -5.6*(3**0.5/2)*horn_dir_2 - 2.75*open_dir_2 + cen, w, 'loss_ll', 'ez')
PPC.Add_Probe(5.6*(3**0.5/2)*x - 2.75*y + cen,\
              5.6*(3**0.5/2)*x + 2.75*y + cen, w, 'loss_R', 'ez')

rod_eps = 0.999*np.ones(91) #Initialize with plasma off (eps = ~1)
rho = PPC.Eps_to_Rho(epsr = rod_eps, plasma = True, w_src = w, wp_max = wpmax) #Initial Parameters
#rho = PPC.Read_Params(output+'/params/'+fname+run_no[0]+'.csv')
#Norms = PPC.Read_Params(output+'/params/'+fname+'_norms.csv')


PPC.Viz_Domain_opt(rho, output+'/plots/'+fname+'_domain.pdf', src_names=['src'],\
     prb_names=['prb1', 'prb2', 'loss_ul', 'loss_ll', 'loss_R'], w=w, wp_max=wpmax,\
         gamma=gamma, uniform=uniform, plasma=True)

In [ ]:
rho_opt, obj, E0, E0l = PPC.Optimize_Waveguide_Penalize(rho, 'src', 'prb1', ['prb2', 'loss_ul', 'loss_ll', 'loss_R'],\
               0.005, 100, plasma = True, wp_max = wpmax, gamma = gamma, uniform = uniform,\
               param_evolution = True, param_out = output+'/run_params')
#               param_evolution = True, param_out = output+'/run_params',\
#               E0 = Norms[0], E0l = Norms[1])

In [ ]:
## Save parameters and visualize ##############################################
PPC.Save_Params(rho_opt, output+'/params/'+fname+run_no[1]+'.csv')
PPC.Save_Params(np.array([E0]+E0l), output+'/params/'+fname+'_norms.csv') 
print(PPC.Rho_to_Eps(rho = rho_opt, plasma = True, w_src = w))
PPC.Params_to_Exp(rho = rho_opt, src = 'src', plasma = True)
PPC.Viz_Sim_abs_opt(rho_opt, ['src'], output+'/plots/'+fname+run_no[1]+'.pdf',\
                    plasma = True, wp_max = wpmax, uniform = uniform, gamma = gamma)
PPC.Save_Params(obj, output+'/plots/'+fname+'_obj'+run_no[1]+'.csv')
PPC.Viz_Obj(obj, output+'/plots/'+fname+'_obj'+run_no[1]+'.pdf')

In [ ]:
[E0]+E0l